# LAB1 - Bài 6 (Titanic): Tiền xử lý dữ liệu thiếu và xóa thuộc tính không cần thiết

**Họ tên:** Đinh Công Vinh Quang  
**MSSV:** 6451071061

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

HO_TEN, MSSV = "Đinh Công Vinh Quang", "6451071061"
print(f"{HO_TEN} - {MSSV}")

Đinh Công Vinh Quang - 6451071061


In [2]:
df = pd.read_csv("titanic.csv")
print("Trước xử lý:"); print(df.isnull().sum()[lambda s: s > 0])
print("Số giá trị khác nhau của ticket:", df["ticket"].nunique())

Trước xử lý:
age           263
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64
Số giá trị khác nhau của ticket: 929


In [3]:
# --- Tiền xử lý (Câu 6) ---
df["deck"] = df["cabin"].str[0]                       # giữ lại ký tự đầu của cabin (boong) trước khi xóa
df = df.drop(columns=["body", "cabin", "boat", "home.dest", "ticket"])

# tuổi: sinh ngẫu nhiên trong [mean - std, mean + std]
rng = np.random.default_rng(42)
mean, std = df["age"].mean(), df["age"].std()
n_null = df["age"].isnull().sum()
df.loc[df["age"].isnull(), "age"] = rng.integers(int(mean - std), int(mean + std) + 1, n_null)

# cổng lên tàu: điền giá trị phổ biến nhất (mode); fare thiếu 1 giá trị: điền trung vị
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"] = df["fare"].fillna(df["fare"].median())
df["deck"] = df["deck"].fillna("U")                   # U = Unknown

In [4]:
print("Sau xử lý, số giá trị rỗng còn lại:", int(df.isnull().sum().sum()))
print(df.columns.tolist())
df.head()

Sau xử lý, số giá trị rỗng còn lại: 0
['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'deck']


,pclass,survived,name,sex,age,sibsp,parch,fare,embarked,deck
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,211.3375,S,B
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,151.5500,S,C
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,151.5500,S,C
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,151.5500,S,C
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,151.5500,S,C


**Giải thích:** xóa `body`, `boat`, `home.dest` (thiếu quá nhiều, không ảnh hưởng đến khả năng sống sót), `ticket` (có quá nhiều giá trị khác nhau, gây nhiễu) và `cabin` (sau khi đã tách ký tự đầu thành `deck`). `age` điền ngẫu nhiên theo mean ± std, `embarked` điền bằng mode ('S'), `fare` thiếu 1 giá trị điền bằng trung vị.